In [1]:
import tensorflow as tf
from pathlib import Path

TRAIN_DIR = Path("../data/raw/classification/PlantVillage/train")
TEST_DIR = Path("../data/raw/classification/PlantVillage/val")  

IMG_SIZE = (224, 224)   # MobileNetV2's native input size
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.15,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.15,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print(f"Classes: {len(class_names)}")
print(f"Training batches: {len(train_ds)}")
print(f"Validation batches: {len(val_ds)}")

Found 43444 files belonging to 38 classes.
Using 36928 files for training.
Found 43444 files belonging to 38 classes.
Using 6516 files for validation.
Classes: 38
Training batches: 1154
Validation batches: 204


In [3]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [4]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
], name="data_augmentation")

In [6]:
IMG_SHAPE = IMG_SIZE + (3,)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SHAPE,
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # freeze for now — head-only training first

inputs = tf.keras.Input(shape=IMG_SHAPE)
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ data_augmentation (Sequential)       │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ true_divide (TrueDivide)             │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ subtract (Subtract)                  │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 38)                  │          48,678 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,306,662 (8.80 MB)

 Trainable params: 48,678 (190.15 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [7]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [9]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class_counts = {cls: len(list((TRAIN_DIR / cls).glob("*"))) for cls in class_names}
counts_array = np.array([class_counts[c] for c in class_names])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(class_names)),
    y=np.repeat(np.arange(len(class_names)), counts_array)
)

class_weight_dict = dict(enumerate(class_weights))
print(class_weight_dict)

{0: np.float64(2.268379281537176), 1: np.float64(2.3049660441426147), 2: np.float64(5.196650717703349), 3: np.float64(0.8687410014397696), 4: np.float64(0.9511340747876347), 5: np.float64(1.3577947243405426), 6: np.float64(1.6714373653431824), 7: np.float64(2.788446726572529), 8: np.float64(1.199646545534876), 9: np.float64(1.4508415709324072), 10: np.float64(1.2306384907370687), 11: np.float64(1.2110838537020518), 12: np.float64(1.0327580468787143), 13: np.float64(1.327831774558347), 14: np.float64(3.3724576929048284), 15: np.float64(0.2595376067865464), 16: np.float64(0.6220147757860375), 17: np.float64(3.9696637426900585), 18: np.float64(1.4344581654890047), 19: np.float64(0.966410108110513), 20: np.float64(1.4290789473684211), 21: np.float64(1.4290789473684211), 22: np.float64(9.448455850369726), 23: np.float64(3.849370902002481), 24: np.float64(0.28076207217454247), 25: np.float64(0.7787896170945073), 26: np.float64(1.2889099863525781), 27: np.float64(3.1408328513591672), 28: np.f

In [10]:
sanity_train = train_ds.take(5)
sanity_val = val_ds.take(2)

history = model.fit(
    sanity_train,
    validation_data=sanity_val,
    epochs=2,
    class_weight=class_weight_dict
)

Epoch 1/2


C:\Users\markp\OneDrive\Desktop\agrisense-ai\ml\venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


5/5 ━━━━━━━━━━━━━━━━━━━━ 16s 2s/step - accuracy: 0.0375 - loss: 4.3238 - val_accuracy: 0.0938 - val_loss: 3.6494
Epoch 2/2
5/5 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.0875 - loss: 3.7253 - val_accuracy: 0.1250 - val_loss: 3.1592
